# Model Testing and Feature Manipulation

In [ ]:
# Import Python Libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [ ]:
# Load and clean data
df = pd.read_excel('airline_ticket_dataset.xlsx')
df_clean = df.dropna()

### Phase 1: The Core Physics of a Ticket (Distance and Volume)
Every flight starts with a basic cost structure. Before we look at competition, we must account for how far the plane is going and how many people are on the route.

In [ ]:
# 1. Create the Log Variables
df_clean['log_fare'] = np.log(df_clean['fare'])
df_clean['log_dist'] = np.log(df_clean['nsmiles'])
df_clean['log_pax'] = np.log(df_clean['passengers'])

# 2. Prepare the Features
# We keep market share and LCC percentage as they are (since they are already 0-1)
X = df_clean[['log_dist', 'log_pax']]
X = sm.add_constant(X)
y = df_clean['log_fare']

# 3. Fit the Log-Log Model
log_model = sm.OLS(y, X).fit()
print(log_model.summary())

We began our analysis by establishing the "baseline" cost of air travel. In the airline industry, distance is the primary driver of cost due to fuel and labor. However, we used logarithmic transformations for both distance and fare. This is important because it acknowledges "economies of scale"—the idea that the cost of flying doesn't increase by a flat dollar amount per mile, but rather tapers off as flights get longer. By also including log_pax, we account for market density, as high-volume routes often enjoy lower per-passenger costs.

### Phase 2: Identifying the "Disruptive Force"
Now that we have the basics, we need to measure how much Low-Cost Carriers (LCCs) actually break the pricing power of major airlines.

In [ ]:
# Variable: Your LCC Metric (Disruptive Force)
# Multiplies the low-fare share by total passengers to find the volume of cheap seats
df_clean['lcc_volume'] = df_clean['lf_ms'] * df_clean['passengers']

# Variable: Airline and Year effects
# We turn these into 1s and 0s so the model knows which carrier is flying the route and which year the ticket was purchased
carrier_dummies = pd.get_dummies(df_clean['carrier_lg'], prefix='Airline').astype(int)
year_dummies = pd.get_dummies(df_clean['Year'], prefix='Year').astype(int)
quarter_dummies = pd.get_dummies(df_clean['quarter'], prefix='q', drop_first=True).astype(int)

# Fit the OLS Regression
X = pd.concat([
    df_clean[['log_dist', 'log_pax', 'lcc_volume']]
], axis=1)
X = sm.add_constant(X) # Add the base intercept
y = df_clean['log_fare']

model = sm.OLS(y, X).fit()
print(model.summary())

To measure competition, we moved beyond simple percentages and created a variable called LCC Volume. A standard 10% market share for a budget airline is just a number, but a 10% share on a route with 10,000 daily passengers represents a massive physical supply of cheap seats. By multiplying the low-fare carrier's market share by the total passenger count, we created an indicator that represents the "Disruptive Force" in a market. This makes sense for this case because it captures the actual pressure placed on legacy carriers to lower their prices to prevent losing thousands of customers to a cheaper neighbor.

### Phase 3: The Monopoly Power

Not all routes are equal. Some are protected by the market dominance of a hub, where an airline can maintain high prices despite competition.

In [ ]:
# Create the Hub Power variable for both endpoints
df_clean['hub_power_1'] = df_clean['large_ms'] * df_clean['TotalPerPrem_city1']
df_clean['hub_power_2'] = df_clean['large_ms'] * df_clean['TotalPerPrem_city2']

# Regress fare on the hub power variables
X = df_clean[['hub_power_1', 'hub_power_2']]
X = sm.add_constant(X)
y = df_clean['log_fare']

model_hub = sm.OLS(y, X).fit()
print(model_hub.summary())

To balance our model, we had to account for Hub Power. In the U.S. aviation market, an airline's pricing power is significantly amplified if they control the "Hub" at either end of the trip. We calculated this as an interaction term between market share and hub status. This indicator is vital because it explains why some monopolies are "expensive monopolies"—they aren't just the biggest carrier; they own the gates and the slots, making it nearly impossible for a low-cost carrier to truly disrupt them.

### Phase 4: Finalizing the Model with Brand and Time Adjustments
We have to recognize that some airlines are just cheaper by design (Brand) and some years were more expensive for everyone (Economy).

In [ ]:
# Variable: Airline and Year effects
# We turn these into 1s and 0s so the model knows which carrier is flying the route and which year the ticket was purchased
carrier_dummies = pd.get_dummies(df_clean['carrier_lg'], prefix='Airline').astype(int)
year_dummies = pd.get_dummies(df_clean['Year'], prefix='Year').astype(int)
quarter_dummies = pd.get_dummies(df_clean['quarter'], prefix='q', drop_first=True).astype(int)

# Fit the OLS Regression
X = pd.concat([
    carrier_dummies, 
    year_dummies, 
    quarter_dummies
], axis=1)
X = sm.add_constant(X) # Add the base intercept
y = df_clean['log_fare']

model = sm.OLS(y, X).fit()
print(model.summary())

Our final narrative concludes by adjusting for Fixed Effects. By including Carrier and Year Dummies, we are "leveling the playing field." These variables act like on/off switches that allow the model to recognize that a ticket on Spirit is naturally cheaper than a ticket on United, regardless of distance. Similarly, the Year Dummies account for macro-economic shifts like fuel inflation. By stripping away these brand and timing biases, we are left with a clear, accurate view of how distance, volume, and competitive disruption actually dictate the price of a ticket in today's market.

### Phase 5: Putting it All Together
In this final stage, we bring every predictor of our feature engineering together to explain the complex reality of airline pricing.

In [ ]:
# Regress fare on all new variables
X = pd.concat([
    df_clean[['lcc_volume', 'log_dist', 'log_pax', 'hub_power_1', 'hub_power_2']], 
    carrier_dummies, 
    year_dummies, 
    quarter_dummies
], axis=1)
X = sm.add_constant(X)
y = df_clean['log_fare']

model_hub = sm.OLS(y, X).fit()
print(model_hub.summary())

This final model represents our full market analysis. By synthesizing every variable, we can explain the complex reality of airline pricing as the result of three primary factors:
- Operational Reality: The cost of the distance and the efficiency of a high-volume route (log_dist and log_pax).
- Competitive Friction: The downward pressure exerted by the physical presence of low-cost seats (lcc_volume).
- Structural Power: The premium an airline can extract through its "Fortress Hub" and market share (hub_power).

Our model shows that airfare is a direct result of the competition between an airline’s structural control and a passenger's available choices. On routes touching highly dominant hubs, we observe a clear price premium. This is because dominant carriers control the essential infrastructure—like gates and flight slots—creating a monopoly effect that allows them to charge more due to limited local competition. In these scenarios, a higher market share for the leading airline almost always translates to higher average fares.

However, we found that Low-Cost Carrier (LCC) Volume acts as the primary equalizer. It is not just the presence of a budget airline that lowers prices, but the actual quantity of seats they offer. As this volume of affordable seats increases, it creates a "price ceiling" that forces even the most dominant airlines to lower their fares to remain competitive.

Ultimately, the market fare is determined by which force is stronger: the Monopoly Power of the hub leader or the Disruptive Force of the low-cost competitors. Our results suggest that the most effective way to lower fares across the industry is to support the expansion of low-cost seat capacity, which remains the only consistent check against the pricing power of dominant hub carriers.

# Data Visualizations

These graphs are meant to aid our understanding of the dataset and the relationship between predictors.

In [ ]:
# --- 1. DATA LOADING & QUALITY CONTROL ---
df = pd.read_excel("airline_ticket_dataset.xlsx")

# Define columns critical for the pricing model
required = [
    "fare", "nsmiles", "passengers", "lf_ms", 
    "large_ms", "TotalPerPrem_city1", "TotalPerPrem_city2", 
    "Year", "quarter", "city1"
]

# Drop missing values and filter for physical/economic reality (no zero values)
df = df.dropna(subset=required).copy()
df = df[(df["fare"] > 0) & (df["nsmiles"] > 0) & (df["passengers"] > 0)]

# --- 2. FEATURE ENGINEERING: CORE PHYSICS ---
# Log-transforming to handle non-linearity and diminish variance
df["log_fare"] = np.log(df["fare"])
df["log_dist"] = np.log(df["nsmiles"])
df["log_pax"]  = np.log(df["passengers"])
df["fare_per_mile"] = df["fare"] / df["nsmiles"]

# --- 3. FEATURE ENGINEERING: MARKET POWER ---
# LCC Volume: The physical supply of budget seats
df["lcc_volume"] = df["lf_ms"] * df["passengers"]

# Hub Power: Interaction between dominant airline share and premium infrastructure
# Represents the "Fortress Hub" effect at both endpoints
df["hub_power_1"] = df["large_ms"] * df["TotalPerPrem_city1"]
df["hub_power_2"] = df["large_ms"] * df["TotalPerPrem_city2"]
df["hub_power"]   = df["hub_power_1"] + df["hub_power_2"]

# --- 4. GLOBAL PLOTTING STYLES ---
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({
    "axes.spines.right": False,
    "axes.spines.top": False,
    "grid.alpha": 0.15,
    "grid.linestyle": "--",
    "figure.facecolor": "none",
    "axes.facecolor": "none"
})

# Setup output directory for high-res deck figures
OUT = "deck_figs"
os.makedirs(OUT, exist_ok=True)

def save(name):
    """Saves plots with high DPI and transparency for presentation decks."""
    plt.savefig(
        os.path.join(OUT, name), 
        dpi=400, 
        bbox_inches="tight", 
        transparent=True
    )

In [ ]:
# --- Plot Aesthetics ---
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({
    "figure.facecolor": "none",
    "axes.facecolor": "none",
    "axes.spines.right": False,
    "axes.spines.top": False,
    "grid.alpha": 0.85,
    "grid.linestyle": "--",
})

fig, ax = plt.subplots(figsize=(11,6))

# --- Data Visualization ---
# Sample to reduce overplotting while maintaining distribution shape
samp = df.sample(6000, random_state=0)

sc = ax.scatter(
    samp["nsmiles"],
    samp["fare"],
    c=samp["fare"],
    cmap="magma",
    alpha=0.6,
    s=20,
    edgecolor="none"
)

# LOWESS smoothing to capture non-linear baseline trend
sns.regplot(
    data=df,
    x="nsmiles",
    y="fare",
    scatter=False,
    lowess=True,
    line_kws={"color":"#111111", "linewidth":3},
    ax=ax
)

# Apply log scale to distance to normalize the distribution
ax.set_xscale("log")

# --- Titles and Labels ---
fig.suptitle(
    "Airfare Increases with Distance",
    fontsize=20,
    fontweight="bold",
    y=0.96,
    x=0.45
)

ax.set_title(
    "Distance explains baseline pricing before competitive effects",
    fontsize=12.5,
    pad=12,
    color="#444444"
)

ax.set_xlabel("Route Distance (miles, log scale)", fontsize=13, labelpad=12)
ax.set_ylabel("Average Fare ($)", fontsize=13, labelpad=12)

# Format y-axis as currency
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))

# --- Colorbar and Layout ---
cbar = fig.colorbar(sc, ax=ax, pad=0.02)
cbar.set_label("Fare ($)", fontsize=11)

sns.despine()
plt.subplots_adjust(top=0.85, bottom=0.15, right=0.90, left=0.10)

# --- Save and Show ---
plt.savefig("02_distance_vs_fare_gradient_clean.png", dpi=300, transparent=True)
plt.show()

This visualization establishes the fundamentals of the aviation market: the relationship between flight distance and ticket price. By applying a logarithmic scale to the x-axis, we reveal that while fares increase with distance, they do so at a diminishing rate—reflecting the operational efficiencies gained on long-haul routes. The LOWESS (Locally Weighted Scatterplot Smoothing) line acts as our "fair value" benchmark; it represents the baseline price consumers should expect to pay based solely on mileage. The vertical spread of data points around this line is the most critical insight for our model—it represents the "Price Variance" that cannot be explained by distance alone. This variance is where strategic factors like Hub Dominance (pushing prices up) and LCC Volume (pulling prices down) exert their influence, shifting the fare away from its operational baseline.

In [ ]:
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({
    "figure.facecolor": "none",
    "axes.facecolor": "none",
    "axes.spines.right": False,
    "axes.spines.top": False,
    "grid.alpha": 0.12,
    "grid.linestyle": "--",
})

fig, ax = plt.subplots(figsize=(11,6))

samp = df.sample(7000, random_state=0).copy()

sc = ax.scatter(
    samp["nsmiles"],
    samp["fare_per_mile"],
    c=samp["fare_per_mile"],   # gradient by $/mile
    cmap="magma",
    s=22,
    alpha=0.55,
    edgecolor="none"
)

sns.regplot(
    data=df,
    x="nsmiles",
    y="fare_per_mile",
    scatter=False,
    lowess=True,
    line_kws={"color":"#111111","linewidth":3.2},
    ax=ax
)

ax.set_xscale("log")

# Title block
fig.text(0.45, 0.94, "Longer Flights Cost Less per Mile",
         ha="center", fontsize=20, fontweight="bold")
fig.text(0.45, 0.90, "Economies of scale show up as declining $/mile with distance",
         ha="center", fontsize=12.5, color="#444444")

ax.set_xlabel("Route Distance (miles, log scale)", fontsize=13, labelpad=12)
ax.set_ylabel("Fare per Mile ($)", fontsize=13, labelpad=12)

# Keep plot clean: cap extreme outliers so it looks tight
ax.set_ylim(0, df["fare_per_mile"].quantile(0.995))

# Colorbar (small + neat)
cbar = fig.colorbar(sc, ax=ax, pad=0.02)
cbar.set_label("Fare per Mile ($)", fontsize=11)

sns.despine()
plt.subplots_adjust(top=0.85, bottom=0.15, right=0.90, left=0.10)

plt.savefig("03_fare_per_mile_gradient.png", dpi=300, transparent=True)
plt.show()

This visualization demonstrates a fundamental principle of airline economics: as flight distance increases, the Fare per Mile decreases significantly. The steep initial drop-off in the LOWESS curve highlights the high fixed costs associated with every flight—such as airport landing fees, gate agent staffing, and the fuel required for takeoff—which are "amortized" over more miles on longer routes. By plotting distance on a logarithmic scale, we can more clearly see that the pricing efficiency gains are most dramatic in the short-haul market (under 1,000 miles). This non-linear relationship is exactly why our final predictive model utilizes a log-log transformation; a simple linear model would fail to capture this "economy of scale," leading to consistent overpricing of long-haul flights and underpricing of short-haul hops.

In [ ]:
# Set aesthetics for the final plot
sns.set_theme(style="whitegrid", context="talk")

plt.rcParams.update({
    "figure.facecolor": "none",
    "axes.facecolor": "none",
    "axes.spines.right": False,
    "axes.spines.top": False,
    "grid.alpha": 0.12,
    "grid.linestyle": "--",
})

fig, ax = plt.subplots(figsize=(11,6))

# Sample to reduce fog
samp = df.sample(7000, random_state=0)

sc = ax.scatter(
    samp["passengers"],
    samp["fare"],
    c=samp["fare"],
    cmap="magma",
    s=22,
    alpha=0.5,
    edgecolor="none"
)

sns.regplot(
    data=df,
    x="passengers",
    y="fare",
    scatter=False,
    lowess=True,
    line_kws={"color":"#111111","linewidth":3.2},
    ax=ax
)

ax.set_xscale("log")
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))

# ---- Clean Title Block ----
fig.text(
    0.5, 0.94,
    "Higher Volume Routes Tend to Be Cheaper",
    ha="center",
    fontsize=20,
    fontweight="bold"
)

fig.text(
    0.5, 0.90,
    "Dense markets exhibit lower average fares",
    ha="center",
    fontsize=12.5,
    color="#444444"
)

ax.set_xlabel("Passengers (log scale)", fontsize=13, labelpad=12)
ax.set_ylabel("Average Fare ($)", fontsize=13, labelpad=12)

# Clean colorbar
cbar = fig.colorbar(sc, ax=ax, pad=0.02)
cbar.set_label("Fare ($)", fontsize=11)

sns.despine()
plt.subplots_adjust(top=0.84, right=0.92)

plt.savefig("04_density_effect_clean.png", dpi=300, transparent=True)
plt.show()

While distance sets the baseline cost, passenger volume determines the market's efficiency. This visualization explores the relationship between market density and average fare. By plotting Passengers on a logarithmic scale, we observe a clear downward trend in pricing as route volume increases. This "Density Effect" is driven by two factors: first, airlines can achieve lower seat-mile costs by utilizing larger, more efficient aircraft on high-traffic routes; and second, dense markets are more likely to attract multiple competitors, which naturally erodes the pricing power of any single carrier. The LOWESS curve highlights that the most significant price drops occur as a market moves from "niche" to "mid-sized" volume, eventually plateauing as the market becomes saturated. Including passenger volume in our OLS model allows us to control for this "economies of density," ensuring that our final coefficients for Hub Power and LCC Volume are not skewed by simple differences in market size.

In [ ]:
# --- 3. DENSITY EFFECT: HIGHER VOLUME ROUTES TEND TO BE CHEAPER ---
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({
    "figure.facecolor": "none",
    "axes.facecolor": "none",
    "axes.spines.right": False,
    "axes.spines.top": False,
    "grid.alpha": 0.12,
    "grid.linestyle": "--",
})

# --- residualize baseline (same as you did) ---
X_base = sm.add_constant(df[["log_dist","log_pax"]])
base = sm.OLS(df["log_fare"], X_base).fit()
df["resid_fare"] = df["log_fare"] - base.predict(X_base)

fig, ax = plt.subplots(figsize=(11,6))

# sample so it's not a fog wall
samp = df.sample(7000, random_state=0)

# gradient points by fare level (optional but looks premium)
sc = ax.scatter(
    samp["lf_ms"],
    samp["resid_fare"],
    c=samp["fare"],        # gradient ties to pricing
    cmap="magma",
    s=20,
    alpha=0.5,
    edgecolor="none"
)

# LOWESS trend
sns.regplot(
    data=df,
    x="lf_ms",
    y="resid_fare",
    scatter=False,
    lowess=True,
    line_kws={"color":"#111111","linewidth":3.2},
    ax=ax
)

# title block
fig.text(0.43, 0.94, "Residual Log(Fare) vs LCC Market Share",
         ha="center", fontsize=20, fontweight="bold")
fig.text(0.43, 0.90, "After controlling for distance and volume, higher LCC share reduces fares",
         ha="center", fontsize=12.5, color="#444444")

ax.set_xlabel("LCC Market Share (lf_ms)", fontsize=13, labelpad=12)
ax.set_ylabel("Residual log(Fare) vs baseline cost", fontsize=13, labelpad=12)
ax.axhline(0, color="#111111", linewidth=1, alpha=0.5)

# clean colorbar (optional)
cbar = fig.colorbar(sc, ax=ax, pad=0.02)
cbar.set_label("Fare ($)", fontsize=11)

sns.despine()
plt.subplots_adjust(top=0.85, bottom=0.15, right=0.85, left=0.15)

plt.savefig("05_lcc_residual_effect_clean.png", dpi=300, transparent=True)
plt.show()

This visualization isolates the impact of competition by plotting the Residual Log(Fare) against Low-Cost Carrier (LCC) Market Share. By using residuals from our baseline "physics" model (which accounts for distance and volume), we have effectively stripped away the operational costs of the flight to see the "pure" effect of market players. The clear downward slope of the LOWESS trend line provides empirical proof that as LCCs capture more of the market, they exert a powerful downward pressure on fares. This isn't just a correlation; it represents the "Competition Discount"—the percentage by which ticket prices drop when legacy carriers are forced to compete with budget-friendly alternatives. This finding is a cornerstone of our final model, confirming that market structure is just as influential as physical distance in determining what a consumer eventually pays.

In [ ]:
# --- Theme / styling (matches your deck) ---
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({
    "figure.facecolor": "none",
    "axes.facecolor": "none",
    "axes.spines.right": False,
    "axes.spines.top": False,
    "grid.alpha": 0.12,
    "grid.linestyle": "--",
})

# --- Ensure ordered quartiles ---
order = ["Low", "Med-Low", "Med-High", "High"]

# If hub_q doesn't exist yet, create it (safe)
if "hub_q" not in df.columns:
    df["hub_q"] = pd.qcut(df["hub_power"], 4, labels=order)

# --- Plot ---
fig, ax = plt.subplots(figsize=(11,6))

# Use an explicit palette list to avoid seaborn future warning
palette = sns.color_palette("mako", n_colors=4)

sns.boxplot(
    data=df,
    x="hub_q",
    y="fare",
    order=order,
    palette=palette,
    showfliers=False,
    linewidth=1.6,
    ax=ax
)

# Optional: make median line stand out a bit more
for line in ax.lines:
    line.set_linewidth(2)

# --- Title block (figure-level, consistent spacing) ---
fig.text(
    0.5, 0.94,
    "Fortress Hubs Extract Price Premiums",
    ha="center",
    fontsize=20,
    fontweight="bold"
)
fig.text(
    0.5, 0.90,
    "Hub Dominance and Average Pricing",
    ha="center",
    fontsize=12.5,
    color="#444444"
)

# --- Labels / formatting ---
ax.set_xlabel("Hub Power Quartile", fontsize=13, labelpad=12)
ax.set_ylabel("Average Fare ($)", fontsize=13, labelpad=12)
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))

# Cleaner grid: y only
ax.grid(axis="y", visible=True)
ax.grid(axis="x", visible=False)

sns.despine()
plt.subplots_adjust(top=0.84, bottom = 0.15)

plt.savefig("06_hub_power_boxplot_clean.png", dpi=300, transparent=True)
plt.show()

This boxplot illustrates the direct correlation between an airline's control over an airport's infrastructure and the prices consumers pay. By dividing our Hub Power metric into quartiles—ranging from competitive, open-access airports ("Low") to dominated "Fortress Hubs" ("High")—we see a clear upward shift in both the median fare and the overall price distribution.

While distance and volume set the floor for pricing, this distribution shows that market dominance sets the ceiling. In high-power quartiles, the lack of viable alternatives allows carriers to extract a premium that exists independently of the flight's operational costs. The increasing height of the boxes as we move from left to right further suggests that as an airport becomes more dominated, pricing becomes not only higher but also more volatile, as the "anchor" of competition is removed. This finding justifies the inclusion of endpoint dominance as a primary variable in our final OLS model, proving that where you fly from is often as important as how far you go.

In [ ]:
# Set aesthetics for the final plot
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({
    "figure.facecolor": "none",
    "axes.facecolor": "none",
    "axes.spines.right": False,
    "axes.spines.top": False,
    "grid.alpha": 0.0,          # no grid on heatmap
})

# --- Bin definitions (same logic, but cleaner) ---
hub_bins = pd.qcut(df["hub_power"], 6, duplicates="drop")
lcc_bins = pd.cut(df["lf_ms"], bins=np.linspace(0, 1, 7), include_lowest=True)

grid = (
    df.groupby([hub_bins, lcc_bins], observed=True)["resid_fare"]
      .mean()
      .unstack()
)

# --- Clean axis labels ---
# Convert interval bins to simple labels like "0–0.17", "0.17–0.33", ...
lcc_labels = [f"{iv.left:.2f}–{iv.right:.2f}" for iv in grid.columns]
hub_labels = [f"{iv.left:.2f}–{iv.right:.2f}" for iv in grid.index]

grid.columns = lcc_labels
grid.index = hub_labels

# --- Rocket-themed diverging cmap (rocket -> light -> rocket reversed)
# This keeps your deck palette but still shows +/- around 0.
cmap = sns.diverging_palette(10, 275, s=85, l=55, as_cmap=True)  # rocket-ish diverging

fig, ax = plt.subplots(figsize=(11,6))

sns.heatmap(
    grid,
    cmap=cmap,
    center=0,
    linewidths=0.6,
    linecolor=(1, 1, 1, 0.6),
    cbar_kws={"label": "Residual log(Fare)"},
    ax=ax
)

# Title block (deck style)
fig.text(0.45, 0.94, "Monopoly vs Competition",
         ha="center", fontsize=20, fontweight="bold")
fig.text(0.45, 0.90, "Hub power raises fares, LCC presence pushes them down (after controlling for cost)",
         ha="center", fontsize=12.5, color="#444444")

ax.set_xlabel("LCC Share Bin (lf_ms)", fontsize=13, labelpad=12)
ax.set_ylabel("Hub Power Quantile", fontsize=13, labelpad=12)

# Clean ticks
ax.set_xticklabels(ax.get_xticklabels(), rotation=0, ha="center")
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

plt.subplots_adjust(top=0.84, right=0.97, bottom = 0.15, left = 0.25)

plt.savefig("07_heatmap_monopoly_vs_competition_rocket.png", dpi=300, transparent=True)
plt.show()

This heatmap visualizes the core conflict at the heart of airfare pricing: the interaction between Hub Power (market dominance at endpoints) and LCC Presence (competitive threat). By plotting the residuals from our baseline model across these two dimensions, we can see the "Net Effect" of market structure on price.

The top-left corner represents the "Hub Dominance" scenario: high market dominance with virtually no budget competition. In these cells, consumers pay the highest "monopoly tax," as indicated by the deep saturation of the heat-map. Conversely, the bottom-right corner represents the "Efficient Market" scenario: low endpoint dominance paired with high LCC volume, resulting in fares significantly below the national average. This visualization proves that an airline's ability to raise prices isn't just a function of its own size, but is strictly capped by the presence of a low-cost alternative. It provides the empirical "why" behind our model's high predictive power, showing that these two variables alone explain the vast majority of price deviations in the U.S. market.

In [ ]:
# Professional styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.family'] = 'sans-serif'

df_copy = df.copy()

df_copy['Period'] = df_copy['Year'].astype(str) + ' Q' + df_copy['quarter'].astype(str)
df_copy['avg_lcc_pct'] = (df_copy['TotalPerLFMkts_city1'] + df_copy['TotalPerLFMkts_city2']) / 2
df_clean = df_copy.dropna(subset=['fare', 'nsmiles', 'avg_lcc_pct', 'large_ms', 'passengers'])

df['avg_lcc_pct'] = (df['TotalPerLFMkts_city1'] + df['TotalPerLFMkts_city2']) / 2
df_clean = df.dropna(subset=['fare', 'nsmiles', 'avg_lcc_pct', 'large_ms', 'passengers'])

# 2. Establish 2022 Q1 Baseline
df_base = df_clean[(df_clean['Year'] == 2022) & (df_clean['quarter'] == 1)]
X_base = sm.add_constant(df_base[['nsmiles', 'avg_lcc_pct', 'large_ms']])
model_base = sm.OLS(df_base['fare'], X_base).fit()

# 3. Loop through each year
for year in sorted(df_clean['Year'].unique()):
    # Get the latest quarter for that year
    latest_q = df_clean[df_clean['Year'] == year]['quarter'].max()
    df_plot = df_clean[(df_clean['Year'] == year) & (df_clean['quarter'] == latest_q)].copy()
    
    # Calculate Residuals
    X_plot = sm.add_constant(df_plot[['nsmiles', 'avg_lcc_pct', 'large_ms']])
    df_plot['residual'] = df_plot['fare'] - model_base.predict(X_plot)
    
    # 4. Plotting
    plt.figure(figsize=(12, 8))
    scatter = sns.scatterplot(
        data=df_plot, x='nsmiles', y='fare', 
        size='passengers', hue='residual', palette='coolwarm', 
        sizes=(40, 600), alpha=0.75, edgecolor='white'
    )
    
    # Label top 5 outliers per year
    top_outliers = df_plot.nlargest(5, 'residual')
    for _, row in top_outliers.iterrows():
        origin = row['city1'].split(',')[0]
        destination = row['city2'].split(',')[0]
        plt.annotate(f"{origin} -> {destination}", (row['nsmiles'], row['fare']), 
                     textcoords="offset points", xytext=(5,5), fontsize=8, fontweight='bold')

    plt.title(f'Airfare Overpricing Frontier: {year} (Q{latest_q})', fontsize=16, fontweight='bold')

These visualizations track the evolution of the U.S. airfare market by comparing yearly pricing structures against a 2022 Q1 baseline. By using the 2022 model as a fixed "yardstick," we can quantify exactly how much specific routes have drifted away from post-pandemic efficiency norms. In this "Frontier" plot, the color gradient represents the Residuals: routes in deep red are the "Overpriced Frontiers"—markets where current fares significantly exceed the predicted price based on distance and volume. The size of each marker denotes passenger volume, which helps us distinguish between expensive "boutique" routes and high-traffic "hub corridors." By automatically labeling the top five outliers, we pinpoint the specific geographic markets where the "Monopoly Tax" is most severe, providing an at-a-glance view of where competitive pressure has failed to stabilize prices.

In [ ]:
# Format axes
plt.figure(figsize=(8,5))

# Sample to reduce overplotting while maintaining distribution shape
plt.hist(df["fare"], bins=75, edgecolor="white", color = "black")

plt.xlabel("Fare ($)")
plt.ylabel("Frequency")

plt.title("Distribution of Airfares")

plt.tight_layout()
plt.savefig("fare_distribution.png", 
            dpi=300, 
            transparent=True)
plt.show()

In [ ]:
# --- 1. Distance-only model ---
X = sm.add_constant(df["log_dist"])
model_dist = sm.OLS(df["log_fare"], X).fit()

df["resid_dist"] = model_dist.resid

# --- 2. Plot ---
sns.set_theme(style="whitegrid", context="talk")

plt.rcParams.update({
    "axes.spines.right": False,
    "axes.spines.top": False,
    "grid.alpha": 0.85,
})

fig, ax = plt.subplots(figsize=(11,6))

# Scatter
ax.scatter(
    df["nsmiles"],
    df["resid_dist"],
    alpha=0.18,
    s=18,
    color="#4B2E83"
)

# Horizontal zero line
ax.axhline(0, color="black", linewidth=2)

# Log scale for distance
ax.set_xscale("log")

# Labels
ax.set_xlabel("Route Distance (miles, log scale)", fontsize=13)
ax.set_ylabel("Residual log(Fare)", fontsize=13)

# Clean descriptive title
ax.set_title(
    "Residual Log(Fare) vs Route Distance",
    fontsize=18,
    pad=18,
    fontweight="bold"
)

sns.despine()
plt.tight_layout()

# --- SAVE DIRECTLY INTO CURRENT NOTEBOOK DIRECTORY ---
plt.savefig(
    "09_residual_after_distance.png",
    dpi=400,
    bbox_inches="tight",
    transparent=True
)

plt.show()

This visualization serves as a crucial diagnostic for our Phase 1 model. By plotting the Residual Log(Fare) against distance after our first regression, we are looking for "randomness." The fact that the data points are balanced evenly above and below the zero-line across all distances (from short-haul to long-haul) proves that our log-log transformation successfully captured the "physics" of mileage.

If we saw a curved or "U-shaped" pattern here, it would mean our model was systematically biased; however, this clean horizontal cloud confirms that distance is no longer the source of error. The remaining vertical spread—the "noise" in this plot—is exactly what we aim to solve in the next phases. By isolating these residuals, we can now confidently move forward to prove that the remaining price differences are driven by human factors: market monopolies and competitive pressure.

## What’s Next for the Analysis

While our current model explains **72.7% of fare variation** using structural market data, we have identified three key areas to evolve this research into a production-ready pricing tool:

### 1. Integrating "Cost-Push" Variables
Currently, our model treats all carriers as having similar cost structures. By integrating **Jet Fuel Spot Prices** and **Labor Contract cycles**, we can distinguish between a fare hike that is "Cost-Push" (airlines passing on expenses) versus "Profit-Pull" (airlines raising prices because they can).

### 2. Infrastructure & Slot Constraints
In the airline industry, a "monopoly" isn't just about market share; it’s about **Gate Control**. We plan to incorporate airport-level data on slot constraints and terminal leases. This will help identify "Fortress Hubs" where new competitors are physically unable to enter, regardless of how high the fares are.

### 3. Real-Time "Fare Fairness" Calculator
We aim to translate our OLS coefficients into a consumer-facing API. By inputting a route and a price, travelers could receive a **"Fairness Score"** based on our model’s predicted log-fare. 
* *Example:* If the actual price is $> 2\sigma$ above our prediction, the tool would advise: *"You are paying a 30% Hub Premium; consider an alternative airport."*

### 4. Post-Pandemic Behavioral Shifts
As business travel patterns continue to stabilize post-2024, we intend to apply **Time-Series Forecasting** to our residuals. This will allow us to detect if the "Monopoly Tax" at major hubs is strengthening or weakening as remote work changes the traditional Monday-Friday demand curves.